# TotalSegmentator inference (nb2): converted NIfTI → segmentations  —  model-specific

Runs TotalSegmentator (v2.18.0) on the GPU VM. Consumes the
**Boundary-A** archive `converted_nifti.tar.lz4` from nb1 and emits the **Boundary-B**
archive `segmentations.tar.lz4` with the canonical layout:
```
<SeriesInstanceUID>/<model>/segmentations/<SeriesInstanceUID>.nii.gz
<SeriesInstanceUID>/<model>/label_map.json      # {label_id: label_name}
```

Supported tasks (selected via the `task` papermill parameter):

- **`total`** (default): 117-structure full-body segmentation (`--ml` multilabel;
  v2 merged the v1 heart chambers into a single `heart` and added 20 new
  structures — sternum, spinal_cord, thyroid_gland, costal_cartilages, ...).
  Label IDs from `class_map['total']`.
- **`lung_vessels`**: task 117, retrained in v2 as 4 classes (`lung_airways`,
  `lung_airways_wall`, `lung_arteries`, `lung_veins`). v2 crops the FOV to the
  lung lobes internally (using the baked 3 mm model), so this is a single call —
  the v1 two-step pre-segmentation pipeline is gone. `--fast` applies only to
  the `total` task. Label IDs from `class_map['lung_vessels']`.

With `--ml`, v2 treats `-o` as the output **file** path (v1 treated it as a
directory). Weights for both tasks (plus the 3 mm crop model) are baked into the
image under `TOTALSEG_HOME_DIR`; nothing is fetched at job runtime.

nb3 (shared) turns the output into DICOM-SEG + radiomics + SR using the
SNOMED mapping rows that match each task's label names.

## Imports

In [ ]:
import json
import shutil
import subprocess
import time
import traceback
from pathlib import Path

NOTEBOOK_START = time.time()
def _elapsed(s=None):
    return f"{time.time() - (s if s is not None else NOTEBOOK_START):.1f}s"
print(f"[T+{_elapsed()}] Imports complete")

## Parameters

In [ ]:
# Boundary-A archive produced by nb1 (local file on the same VM).
converted_nifti_path = "converted_nifti.tar.lz4"

# Short model identifier used in the Boundary-B layout (<uid>/<model>/...).
model_name = "total"

# 'cuda' for GPU, 'cpu' for CPU-only.
accelerator = "cuda"

# Optional GCS prefix (gs://bucket/prefix) for checkpoint/resume on preemption: each
# finished series' output is saved there and a retried VM skips it. run_id (the
# Cromwell workflow id, from the WDL) namespaces the checkpoint. Empty = disabled.
checkpoint_gcs = ""
run_id = ""

# Model-specific knobs injected via `papermill -f inference_params.yaml`.
# fast=True uses TotalSegmentator's 3mm model (faster, lower resolution);
# only valid for task='total'.
fast = False

# TotalSegmentator task: 'total' (default, 117 structures in v2) or
# 'lung_vessels' (4 classes in v2: airways, airway walls, arteries, veins).
task = "total"

## Extract Boundary-A archive

In [ ]:
NIFTI_DIR = Path('/tmp/converted_nifti')
SEG_DIR = Path('/tmp/segmentations')
for _d in (NIFTI_DIR, SEG_DIR):
    if _d.exists():
        shutil.rmtree(_d)
    _d.mkdir(parents=True, exist_ok=True)

subprocess.run(f'lz4 -d -c {converted_nifti_path} | tar -xf - -C {NIFTI_DIR.parent}',
               shell=True, check=True)
if (NIFTI_DIR / 'converted_nifti').is_dir():
    NIFTI_DIR = NIFTI_DIR / 'converted_nifti'
series_uids = sorted(d.name for d in NIFTI_DIR.iterdir() if d.is_dir())
print(f'Series : {len(series_uids)}  |  task={task}  fast={fast}')

# ---- checkpoint / resume (segmentator_checkpoint.py is fetched next to the notebook by
#      the WDL; no-op when checkpoint_gcs is empty) ----
import os, sys
for _p in (os.getcwd(), str(Path.cwd())):
    if _p not in sys.path:
        sys.path.insert(0, _p)
try:
    from segmentator_checkpoint import Checkpointer
except ImportError:
    Checkpointer = None
if checkpoint_gcs and Checkpointer is None:
    raise RuntimeError('checkpoint_gcs is set but segmentator_checkpoint.py was not found '
                       'next to the notebook (the WDL fetches it from gitRepo/gitBranch)')
ckpt = Checkpointer(checkpoint_gcs, run_id) if Checkpointer else None
completed_seg = ckpt.restore_segs(SEG_DIR) if ckpt else set()
if completed_seg:
    print(f'[T+{_elapsed()}] {len(completed_seg)} series restored from checkpoint')

## Authoritative label map from TotalSegmentator's class map

In [ ]:
from totalsegmentator.map_to_binary import class_map

VALID_TASKS = ('total', 'lung_vessels')
if task not in VALID_TASKS:
    raise ValueError(f"Unknown task '{task}'; valid tasks: {VALID_TASKS}")

TASK_LABELS = {str(k): v for k, v in class_map[task].items()}
print(f'Loaded {len(TASK_LABELS)} TotalSegmentator label ids for task={task}')

## Run TotalSegmentator → Boundary-B layout

In [ ]:
errors = []
usage_metrics = {'series': {}}

def _run_task(nii, work, task, fast):
    """One TotalSegmentator v2 call producing a multilabel volume. With --ml, -o is
    the output FILE path. Subtasks (lung_vessels) crop their FOV internally, so
    every task is a single invocation; --fast is only meaningful for 'total'.

    NOTE: nnunetv2's preprocessing workers pass torch tensors through /dev/shm.
    Cromwell's GCP Batch backend sizes /dev/shm proportional to VM RAM, so this
    is fine on Terra; a local `docker run` needs --shm-size (the docker default
    of 64 MB crashes with "unable to allocate shared memory")."""
    out_file = work / 'seg.nii.gz'
    cmd = ['TotalSegmentator', '-i', str(nii), '-o', str(out_file), '--ml']
    if task != 'total':
        cmd += ['--task', task]
    if fast:
        if task != 'total':
            raise ValueError(f"fast=True is only supported for task 'total' (got {task!r})")
        cmd.append('--fast')
    print(f'  {" ".join(cmd)}', flush=True)
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0:
        raise RuntimeError(f'TotalSegmentator rc={res.returncode}\n{res.stderr}')
    if not out_file.exists():
        # Defensive: some versions write .nii when handed a .nii.gz suffix or
        # vice versa — accept any NIfTI the call produced in the work dir.
        cands = sorted(work.glob('seg.nii*'))
        if not cands:
            raise RuntimeError('no multilabel NIfTI produced')
        out_file = cands[0]
    return out_file

for uid in series_uids:
    nii = NIFTI_DIR / uid / f'{uid}.nii.gz'
    if not nii.exists():
        cands = list((NIFTI_DIR / uid).glob('*.nii.gz'))
        if not cands:
            errors.append(f'{uid}: no NIfTI found')
            continue
        nii = cands[0]
    if (uid, model_name) in completed_seg:
        if not (SEG_DIR / uid / 'reference.nii.gz').exists():
            shutil.copy(str(nii), str(SEG_DIR / uid / 'reference.nii.gz'))
        usage_metrics['series'][uid] = {'model_inference_s': None, 'checkpoint_restored': True}
        print(f'[T+{_elapsed()}] {uid}: restored from checkpoint')
        continue
    work = Path('/tmp/ts_work') / uid
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True, exist_ok=True)
    print(f'[T+{_elapsed()}] {uid} (task={task}):', flush=True)
    t0 = time.time()
    try:
        dest = SEG_DIR / uid / model_name / 'segmentations'
        dest.mkdir(parents=True, exist_ok=True)

        produced = _run_task(nii, work, task, fast)
        target = dest / (uid + '.nii.gz')
        if produced.name.endswith('.gz'):
            shutil.move(str(produced), str(target))
        else:
            subprocess.run(f'gzip -c "{produced}" > "{target}"', shell=True, check=True)

        (SEG_DIR / uid / model_name / 'label_map.json').write_text(
            json.dumps({'model': model_name, 'labels': TASK_LABELS}, indent=2))
        shutil.copy(str(nii), str(SEG_DIR / uid / 'reference.nii.gz'))
        usage_metrics['series'][uid] = {'model_inference_s': round(time.time() - t0, 1)}
        print(f'  done in {usage_metrics["series"][uid]["model_inference_s"]}s')
        if ckpt:
            ckpt.save_seg(SEG_DIR, uid, model_name)
    except Exception as exc:
        errors.append(f'{uid}: {traceback.format_exc()}')
        print(f'  ERROR: {exc}')
    finally:
        shutil.rmtree(work, ignore_errors=True)

if errors:
    Path('inference_errors.txt').write_text('\n'.join(errors))
print(f'[T+{_elapsed()}] Inference complete ({len(errors)} error(s))')

## Package Boundary-B archive + usage metrics

In [ ]:
import csv
produced = [d for d in SEG_DIR.iterdir() if d.is_dir()]
if not produced:
    raise RuntimeError('No segmentations produced — see inference_errors.txt')

subprocess.run(f'tar -cf - -C {SEG_DIR.parent} {SEG_DIR.name} | lz4 > segmentations.tar.lz4',
               shell=True, check=True)
size_mb = Path('segmentations.tar.lz4').stat().st_size / (1024 ** 2)

usage_metrics['total_elapsed_s'] = round(time.time() - NOTEBOOK_START, 1)
with open('inference_UsageMetrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['SeriesInstanceUID', 'model', 'model_inference_s', 'run_total_elapsed_s',
                'checkpoint_restored'])
    for uid, m in usage_metrics['series'].items():
        w.writerow([uid, model_name, m.get('model_inference_s') if m.get('model_inference_s') is not None else '',
                    usage_metrics['total_elapsed_s'], bool(m.get('checkpoint_restored', False))])

# Inference finished and the archive is written: the checkpoint is no longer needed.
if ckpt:
    ckpt.cleanup()

print(f'[T+{_elapsed()}] Wrote segmentations.tar.lz4 ({size_mb:.1f} MB, {len(produced)} series)')